In [18]:
import os
import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, DecimalType, TimestampType
)
from pyspark.sql.functions import col, count, when, isnull, lower, trim, to_date, row_number, max, min, upper

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)


Spark ilə MinIO-dakı satış faylını oxuyun. Sxemi, sətir sayını və ilk 10 sətri göstərin. Vaxt sütununun tipinin düzgün təyin olunduğunu yoxlayın.

In [2]:
minio_path = "s3a://retail/raw/sales_2026-10-21.csv"

df_sales = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(minio_path)
)

df_sales.printSchema()
df_sales.show(10, truncate=False)
total_rows = df_sales.count()
print(f"Total rows in the DataFrame: {total_rows}")

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sale_ts: timestamp (nullable = true)

+--------+--------+----------+--------+----------+-------------------+
|sale_id |store_id|product_id|quantity|unit_price|sale_ts            |
+--------+--------+----------+--------+----------+-------------------+
|SL000420|S02     |P021      |2       |1.62      |2026-10-21 09:02:15|
|SL001090|S02     |P008      |5       |3.8       |2026-10-21 09:02:53|
|SL001894|S06     |P021      |1       |1.8       |2026-10-21 09:03:46|
|SL001007|S06     |P002      |2       |1.52      |2026-10-21 09:04:22|
|SL001406|S02     |P013      |2       |1.89      |2026-10-21 09:04:46|
|SL001930|S11     |P007      |2       |2.16      |2026-10-21 09:05:54|
|SL001283|S01     |P026      |4       |6.2       |2026-10-21 09:06:09|
|SL000849|S02     |P026      |4       

Miqdarı boş və ya mənfi olan sətirləri silin. Hər satış üçün ümumi məbləğ (miqdar × qiymət) sütunu yaradın.

In [3]:

df_cleaned = df_sales.filter(col("quantity").isNotNull() & (col("quantity") > 0))

df_cleaned = df_sales.filter(
    col("quantity").isNotNull() & (col("quantity") > 0)
)

df_cleaned = df_cleaned.withColumn(
    "total_amount",
    col("quantity") * col("unit_price")
)

df_cleaned.show(5)          



+--------+--------+----------+--------+----------+-------------------+------------+
| sale_id|store_id|product_id|quantity|unit_price|            sale_ts|total_amount|
+--------+--------+----------+--------+----------+-------------------+------------+
|SL000420|     S02|      P021|       2|      1.62|2026-10-21 09:02:15|        3.24|
|SL001090|     S02|      P008|       5|       3.8|2026-10-21 09:02:53|        19.0|
|SL001894|     S06|      P021|       1|       1.8|2026-10-21 09:03:46|         1.8|
|SL001007|     S06|      P002|       2|      1.52|2026-10-21 09:04:22|        3.04|
|SL001406|     S02|      P013|       2|      1.89|2026-10-21 09:04:46|        3.78|
+--------+--------+----------+--------+----------+-------------------+------------+
only showing top 5 rows



Satış vaxtından saat məlumatını çıxarıb hər saat üzrə satış sayını və ümumi satış məbləğini hesablayın. Ən çox gəlir gətirən saatı müəyyən edin.


In [4]:
df_cleaned.createOrReplaceTempView("df_cleaned")

hourly_sales = spark.sql("""
    SELECT
        HOUR(sale_ts) AS sale_hour,
        COUNT(*) AS total_sales_count,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM df_cleaned
    GROUP BY HOUR(sale_ts)
    ORDER BY total_revenue DESC
    LIMIT 24
""")

hourly_sales.show()

+---------+-----------------+-------------+
|sale_hour|total_sales_count|total_revenue|
+---------+-----------------+-------------+
|       19|              221|      1718.85|
|       14|              203|      1623.56|
|       12|              198|      1416.17|
|       18|              204|      1367.74|
|       13|              209|      1291.58|
|       20|              151|      1197.62|
|       11|              143|       971.38|
|       16|              120|       935.64|
|       17|              148|       868.91|
|       15|              120|       812.31|
|        9|               69|       674.79|
|       10|               98|       663.15|
|       21|               72|       397.04|
+---------+-----------------+-------------+



Təmizlənmiş satış datasını mağaza identifikatoruna görə partisiyalayaraq MinIO-da processed/ qovluğuna parquet formatında yazın. Sonra yalnız bir mağazanın datasını geri oxuyun.

In [6]:
minio_processed_path = "s3a://retail/processed/hourly_sales_summary.csv"

df_cleaned.write \
    .mode("overwrite") \
    .partitionBy("store_id") \
    .parquet(minio_processed_path)

PostgreSQL-də stores və products cədvəllərini yaradın, CSV datasını Spark ilə ora yükləyi

In [7]:
minio_path_stores = "s3a://retail/raw/stores.csv"
minio_path_products = "s3a://retail/raw/products (1).csv"

df_stores = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(minio_path_stores)
)

df_products = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(minio_path_products)
)
df_stores.show(5)
df_products.show(5)

+--------+--------------+----+--------+
|store_id|    store_name|city|  region|
+--------+--------------+----+--------+
|     S01|   Baku Nizami|Baku|Absheron|
|     S02|  Baku 28 Mall|Baku|Absheron|
|     S03|  Baku Yasamal|Baku|Absheron|
|     S04|Baku Narimanov|Baku|Absheron|
|     S05| Baku Sabunchu|Baku|Absheron|
+--------+--------------+----+--------+
only showing top 5 rows

+----------+------------+--------+
|product_id|product_name|category|
+----------+------------+--------+
|      P001|       Bread|    Food|
|      P002|        Milk|    Food|
|      P003|      Cheese|    Food|
|      P004|        Eggs|    Food|
|      P005|        Rice|    Food|
+----------+------------+--------+
only showing top 5 rows



In [8]:
JDBC_URL = "jdbc:postgresql://postgres:5432/lesson23" 

PG_USER = "matrix"
PG_PASSWORD = "matrix123" 
PG_DRIVER = "org.postgresql.Driver"

df_products.write.jdbc(
    url=JDBC_URL,
    table="products",
    mode="overwrite",
    properties={
        "user": PG_USER, 
        "password": PG_PASSWORD,
        "driver": PG_DRIVER, 
        "batchsize": "5000"
    }
)


df_stores.write.jdbc(
    url=JDBC_URL,
    table="public.stores",
    mode="overwrite",
    properties={
        "user": PG_USER, 
        "password": PG_PASSWORD,
        "driver": PG_DRIVER, 
        "batchsize": "5000"
    }
)

print("Data written to PostgreSQL successfully.")

Data written to PostgreSQL successfully.


PostgreSQL-dən stores cədvəlini Spark-a oxuyun və yalnız Bakı mağazalarını göstərin.


In [9]:
jdbc_url = "jdbc:postgresql://postgres:5432/lesson23"
jdbc_properties = {
    "user": PG_USER,
    "password": PG_PASSWORD,
    "driver": PG_DRIVER
}

df_stores_pg = spark.read \
    .jdbc(url=jdbc_url, table="stores", properties=jdbc_properties)

df_baku_stores = df_stores_pg.filter(col("city") == "Baku")

df_baku_stores.show(truncate=False)

+--------+--------------+----+--------+
|store_id|store_name    |city|region  |
+--------+--------------+----+--------+
|S01     |Baku Nizami   |Baku|Absheron|
|S02     |Baku 28 Mall  |Baku|Absheron|
|S03     |Baku Yasamal  |Baku|Absheron|
|S04     |Baku Narimanov|Baku|Absheron|
|S05     |Baku Sabunchu |Baku|Absheron|
+--------+--------------+----+--------+



MinIO-dakı satış datasını PostgreSQL-dəki stores və products cədvəlləri ilə birləşdirin. Hər satışın yanında mağaza adı, şəhər, məhsul adı və kateqoriya görünsün.

In [10]:
df_sales_cleaned = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .parquet("s3a://retail/processed/hourly_sales_summary.csv")

df_stores = spark.read.jdbc(url=jdbc_url, table="stores", properties=jdbc_properties)
df_products = spark.read.jdbc(url=jdbc_url, table="products", properties=jdbc_properties)

df_joined = (
    df_sales_cleaned
    .join(df_stores, on="store_id", how="inner")
    .join(df_products, on="product_id", how="inner")
)

df_sales_full = df_joined.select(
    "product_id",
    "sale_id",
    "sale_ts",
    "store_name",
    "city",
    "product_name",
    "category",
    "quantity",
    "unit_price",
    "total_amount"
)

df_sales_full.show(10, truncate=False)

+----------+--------+-------------------+-----------+----+------------+--------+--------+----------+------------------+
|product_id|sale_id |sale_ts            |store_name |city|product_name|category|quantity|unit_price|total_amount      |
+----------+--------+-------------------+-----------+----+------------+--------+--------+----------+------------------+
|P001      |SL000194|2026-10-21 21:44:19|Baku Nizami|Baku|Bread       |Food    |6       |0.8       |4.800000000000001 |
|P001      |SL001123|2026-10-21 21:28:27|Baku Nizami|Baku|Bread       |Food    |1       |0.8       |0.8               |
|P001      |SL001152|2026-10-21 20:25:00|Baku Nizami|Baku|Bread       |Food    |5       |0.64      |3.2               |
|P001      |SL001957|2026-10-21 19:59:09|Baku Nizami|Baku|Bread       |Food    |1       |0.8       |0.8               |
|P001      |SL000635|2026-10-21 19:41:43|Baku Nizami|Baku|Bread       |Food    |3       |0.8       |2.4000000000000004|
|P001      |SL001484|2026-10-21 19:27:34

Şəhər üzrə və kateqoriya üzrə ümumi satış məbləğini ayrı-ayrı hesablayın, hər ikisini azalan sırada göstərin.

In [11]:
df_sales_full.createOrReplaceTempView("sales_full")

df_city_revenue = spark.sql("""
    SELECT 
        city, 
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM sales_full
    GROUP BY city
    ORDER BY total_revenue DESC
""")

df_city_revenue.show(truncate=False)

df_category_revenue = spark.sql("""
    SELECT 
        category, 
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM sales_full
    GROUP BY category
    ORDER BY total_revenue DESC
""")

df_category_revenue.show(truncate=False)

+-----------+-------------+
|city       |total_revenue|
+-----------+-------------+
|Baku       |8870.73      |
|Sumgait    |1960.93      |
|Ganja      |1587.89      |
|Mingachevir|672.41       |
|Lankaran   |549.39       |
|Sheki      |297.39       |
+-----------+-------------+

+-------------+-------------+
|category     |total_revenue|
+-------------+-------------+
|Food         |3615.95      |
|Personal Care|2907.49      |
|Beverages    |2761.7       |
|Electronics  |2034.45      |
|Household    |1952.26      |
|Stationery   |666.89       |
+-------------+-------------+



Hər mağaza üzrə satış sayını və ümumi satış məbləğini PostgreSQL-də store_sales_summary adlı cədvələ yazın.


In [12]:
from pyspark.sql.functions import count, sum, round, col

df_store_summary = df_sales_full.groupBy("store_name") \
    .agg(
        count("sale_id").alias("total_sales_count"),
        round(sum("total_amount"), 2).alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc())

df_store_summary.show(truncate=False)

df_store_summary.write \
    .jdbc(url=jdbc_url, table="store_sales_summary", mode="overwrite", properties=jdbc_properties)

+------------------+-----------------+-------------+
|store_name        |total_sales_count|total_revenue|
+------------------+-----------------+-------------+
|Baku Nizami       |327              |2511.77      |
|Baku 28 Mall      |259              |1963.53      |
|Baku Narimanov    |221              |1767.91      |
|Baku Sabunchu     |195              |1471.63      |
|Sumgait Center    |173              |1273.23      |
|Baku Yasamal      |197              |1155.89      |
|Ganja Center      |131              |903.85       |
|Sumgait Mall      |107              |687.7        |
|Ganja Kapaz       |118              |684.04       |
|Mingachevir Center|89               |672.41       |
|Lankaran Center   |87               |549.39       |
|Sheki Center      |52               |297.39       |
+------------------+-----------------+-------------+



Ən çox gəlir gətirən 5 məhsulu tapın və nəticəni PostgreSQL-də top_products cədvəlinə yazın.


In [13]:
df_top_products = df_sales_full.groupBy("product_id", "product_name") \
    .agg(round(sum("total_amount"), 2).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .limit(5)

df_top_products.show(truncate=False)

df_top_products.write \
    .jdbc(url=jdbc_url, table="top_products", mode="overwrite", properties=jdbc_properties)

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|P003      |Cheese      |1094.6       |
|P009      |Coffee      |1017.52      |
|P026      |Shampoo     |869.24       |
|P004      |Eggs        |843.84       |
|P012      |Detergent   |811.8        |
+----------+------------+-------------+



Hər mağaza üçün ən çox gəlir gətirən məhsulu tapın.


In [14]:
df_sales_full.createOrReplaceTempView("sales_full")

df_top_product_per_store = spark.sql("""
    WITH store_product_revenue AS (
        SELECT 
            store_name,
            product_name,
            ROUND(SUM(total_amount), 2) AS product_revenue
        FROM sales_full
        GROUP BY store_name, product_name
    ),
    ranked_products AS (
        SELECT 
            store_name,
            product_name,
            product_revenue,
            ROW_NUMBER() OVER (
                PARTITION BY store_name 
                ORDER BY product_revenue DESC
            ) AS row_num
        FROM store_product_revenue
    )
    SELECT 
        store_name,
        product_name,
        product_revenue
    FROM ranked_products
    WHERE row_num = 1
    ORDER BY product_revenue DESC
""")

df_top_product_per_store.show(truncate=False)

+------------------+------------+---------------+
|store_name        |product_name|product_revenue|
+------------------+------------+---------------+
|Baku Nizami       |Coffee      |244.9          |
|Baku 28 Mall      |Detergent   |205.92         |
|Baku Narimanov    |Headphones  |168.0          |
|Baku Sabunchu     |Detergent   |156.42         |
|Sumgait Center    |Coffee      |120.87         |
|Ganja Center      |Cream       |117.86         |
|Baku Yasamal      |Detergent   |87.12          |
|Ganja Kapaz       |Cheese      |80.6           |
|Sumgait Mall      |Cheese      |79.3           |
|Mingachevir Center|Shampoo     |73.16          |
|Sheki Center      |Cheese      |51.35          |
|Lankaran Center   |Milk        |48.07          |
+------------------+------------+---------------+



Kateqoriya üzrə gəlir hesabatını MinIO-ya həm parquet, həm CSV formatında yazın və fayl ölçülərini müqayisə edin.


In [15]:
df_category_report = df_sales_full.groupBy("category") \
    .agg(round(sum("total_amount"), 2).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

parquet_path = "s3a://retail/reports/category_revenue_parquet"
csv_path = "s3a://retail/reports/category_revenue_csv"

df_category_report.write.mode("overwrite").option("header", "true").parquet(parquet_path)
df_category_report.write.mode("overwrite").option("header", "true").csv(csv_path)
print("Category revenue report written to MinIO in Parquet and CSV formats successfully.")

Category revenue report written to MinIO in Parquet and CSV formats successfully.


Satış datasını Spark SQL-də görünüşə çevirin və SQL sorğusu ilə hər şəhərin orta satış məbləğini tapın.

In [16]:
df_sales_full.createOrReplaceTempView("sales_view")

df_avg_sales_per_city = spark.sql("""
    SELECT 
        city,
        ROUND(AVG(total_amount), 2) AS avg_sale_amount
    FROM sales_view
    GROUP BY city
    ORDER BY avg_sale_amount DESC
""")

df_avg_sales_per_city.show(truncate=False)

+-----------+---------------+
|city       |avg_sale_amount|
+-----------+---------------+
|Mingachevir|7.56           |
|Baku       |7.4            |
|Sumgait    |7.0            |
|Ganja      |6.38           |
|Lankaran   |6.31           |
|Sheki      |5.72           |
+-----------+---------------+



PostgreSQL-dəki store_sales_summary cədvəlinin ümumi məbləğinin MinIO-dakı təmizlənmiş satış datasının ümumi məbləği ilə eyni olduğunu təsdiqləyin.

In [22]:
df_pg_summary = spark.read.jdbc(url=jdbc_url, table="store_sales_summary", properties=jdbc_properties)
pg_total_revenue = df_pg_summary.select(round(sum("total_revenue"), 2)).collect()[0][0]

minio_total_revenue = df_sales_full.select(round(sum("total_amount"), 2)).collect()[0][0]

print(f"PostgreSQL (store_sales_summary) Ümumi Məbləğ : {pg_total_revenue} AZN")
print(f"MinIO (Təmizlənmiş Sales Data) Ümumi Məbləğ  : {minio_total_revenue} AZN")


PostgreSQL (store_sales_summary) Ümumi Məbləğ : 13938.74 AZN
MinIO (Təmizlənmiş Sales Data) Ümumi Məbləğ  : 13938.74 AZN


In [23]:
spark.stop()